# Stage 2 Notebook 38 - Exp2GG Full BDD curve dataset

**The elephant in the room.** Across 30+ Exp2 variants, every notebook has used `LIMIT_TRAIN = 3000`. This caps each epoch to the FIRST 3000 samples. The full BDD curve dataset is much larger -- I just confirmed in `BDDJointCurveDataset.__init__` (train_joint_model_experiment.py:262): when `limit > 0` it slices `items[:limit]`, otherwise all items are used.

Comparison to published lane detectors:
- CLRKDNet: trains on CULane (88K samples) for 70 epochs ~= 6.2M sample-passes total.
- Our Exp2 series: 3000 samples * 20 epochs = 60K sample-passes.
- **We've been training with 100x less compute.** No wonder the architecture's capacity is undertrained.

Six iterations on top of LaneQueryHead (mask aux, lambda fix, cosine LR, KD) lifted decoded_f1 from 0.027 to 0.047 (+74%). The plateau isn't structural training instability anymore -- it's data scarcity.

Exp2GG = Exp2EE recipe + `--limit-train 0` (= no cap). At ~30K samples per epoch, this is 10x more data per epoch. With cosine LR over 15 epochs, total sample-passes = ~450K -- 7.5x our previous best. Closer to the territory CLRKDNet trains in.

Single notebook-arg change: `LIMIT_TRAIN = 3000 -> 0`. Plus: `--print-every: 5 -> 50` (the per-step prints would otherwise dominate the log at ~3,750 steps/epoch instead of 375), `val_batches: 40 -> 80` (more val samples for more reliable metric).

**Compute budget**: ~10x more samples per epoch means ~10x slower epochs. Estimate ~10 min per epoch on Colab Pro GPU vs ~1 min for the 3000-sample runs. 15 epochs ~= 2.5 hours. Acceptable.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 15-epoch full-dataset run (~2.5 hours).
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint_smoke.log
OK exp33_rmt_gca_mask_uncertainty_full_dataset_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=3.0678 det_loss=3.5754 grad_cos=0.2690 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.49788224697113037, 'gate/lane_mean': 0.50132817029953, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full15'
    EPOCHS = 15
    BATCH_SIZE = 8
    LIMIT_TRAIN = 0   # 0 = no cap, use full dataset (the key change)
    LIMIT_VAL = 0     # also use full val split for reliable metrics
    PRINT_EVERY = 50  # less verbose with ~10x more steps per epoch

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, '(0 = full dataset)', flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: 0 (0 = full dataset)
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp33_rmt_gca_mask_uncertainty_full_dataset_joint_full15 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint_full15.tar --epochs 15 --batch-size 8 --limit-train 0 --limit-val 0 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint_full15.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint_full15_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp33_rmt_

0

## What to watch in Exp2GG training

Pass criteria at epoch 15:
- **`samples=N`** in the dataset log: should report N >> 3000 (likely 25K-35K). This confirms full dataset is in use.
- **`val/lane/decoded_f1 >= 0.10`**: 2x the Exp2EE plateau. With 10x more data, decoded ranking should improve substantially.
- **`val/matched_line_iou >= 0.20`**: geometry quality jumps with more lane variations seen.
- **`val/lane/decoded_oracle_f1 >= 0.15`**: oracle ceiling lifts as backbone learns better lane features.
- No late-epoch collapse (cosine LR active).

If decoded_f1 plateaus at 0.05 even with 10x more data: data scarcity ISN'T the bottleneck. The architecture itself is the limit. Pivot to: bigger backbone (ResNet-50 ImageNet pretrained), or higher resolution (480x800 or 720x1280), or external CLRKDNet teacher.

If decoded_f1 jumps to >= 0.10: data scarcity WAS the bottleneck. We can scale further: 30 epochs on full data, or larger model.